In [2]:
import torch
import triton
import triton.language as tl

In [3]:
@triton.jit
def add_vectors(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE):
    pid = tl.program_id(axis=0) # Get the current block index in the x-axis (must do for every axis that we are dealing with, 1D only in this case)
    block_start = pid * BLOCK_SIZE # Get the set of threads that are involved in this current run
    offsets = tl.arange(block_start, block_start + BLOCK_SIZE) # Get the threads involved in this operation
    x = x_ptr + offsets # Get all the values in the corresponding range for the threads
    y = y_ptr + offsets
    # Masking, important for the last block, if n_elements is not divisible by block_size, then there will be more threads active than elements. So this prevents the extra threads from running
    # E.g. if block_size = 8, grid_size = 4 blocks (so 32 threads), n_elements = 28
    # Last block mask (indices 24-31): True, True, True, True, False, False, False, False -> So no operations done on indices 28-31
    mask = offsets < n_elements 
    tl.load(x, mask = mask)
    tl.load(y, mask = mask)
    output = x + y
    tl.store(out_ptr + offsets, output, mask = mask)

In [26]:
@triton.jit
def matrix_multiply(A_ptr, B_ptr, stride_A_row, stride_A_col, stride_B_row, stride_B_col, out_ptr, stride_out_row, stride_out_col, M, N, P, BLOCK_SIZE_ROW: tl.constexpr, BLOCK_SIZE_COL:tl.constexpr, BLOCK_SIZE_K: tl.constexpr):
    idx_row = tl.program_id(axis = 0)
    idx_col = tl.program_id(axis = 1)
    offset_out_row = idx_row * BLOCK_SIZE_ROW + tl.arange(0, BLOCK_SIZE_ROW)
    offset_out_column = idx_col * BLOCK_SIZE_COL + tl.arange(0, BLOCK_SIZE_COL)
    out_ptrs = out_ptr + offset_out_row[:, None] * stride_out_row + offset_out_column[None,:] * stride_out_col
    out_mask = (offset_out_row[:, None] < M) & (offset_out_column[None, :] < P)
    block = tl.zeros((BLOCK_SIZE_ROW, BLOCK_SIZE_COL), dtype=tl.float32)
    for K in range(0, N, BLOCK_SIZE_K):
        # Set of rows to load from A (BLOCK_SIZE_ROW, N)
        offset_A_row = idx_row * BLOCK_SIZE_ROW + tl.arange(0, BLOCK_SIZE_ROW)
        offset_A_col = K + tl.arange(0, BLOCK_SIZE_K)
        A_ptrs = A_ptr + offset_A_row[:, None] * stride_A_row + offset_A_col[None,:] * stride_A_col
        # Set of rows to load from B (N, BLOCK_SIZE_COL) 
        offset_B_row = K + tl.arange(0, BLOCK_SIZE_K) 
        offset_B_col = idx_col * BLOCK_SIZE_COL + tl.arange(0, BLOCK_SIZE_COL)
        B_ptrs = B_ptr + offset_B_row[:, None] * stride_B_row + offset_B_col[None,:] * stride_B_col
        A = tl.load(A_ptrs, mask = (offset_A_row[:, None] < M) & (offset_A_col[None,:] < N))
        B = tl.load(B_ptrs, mask = (offset_B_col[None,:] < N) & (offset_B_col[None,:] < P))
        block += tl.dot(A, B)
        tl.store(out_ptrs, block, mask = out_mask)




In [ ]:
def test_matmul():
    # Dimensions for the test
    M = 4  # number of rows in A
    N = 5  # number of columns in A and rows in B
    P = 3  # number of columns in B
    def next_power_of_2(n: int) -> int:
        """Return the smallest power of 2 >= n."""
        if n <= 1:
            return 1
        return 1 << (n - 1).bit_length()
    # Generate random matrices A (M x N) and B (N x P)
    A = torch.randn((M, N), dtype=torch.float32, device='cuda')
    B = torch.randn((N, P), dtype=torch.float32, device='cuda')
    # Zero matrix for output (M x P)
    out = torch.zeros((M, P), dtype=torch.float32, device='cuda')
    BLOCK_SIZE_ROW = 16
    BLOCK_SIZE_COL = 16
    BLOCK_SIZE_K = 16
    grid = (
        triton.cdiv(M, BLOCK_SIZE_ROW),  # number of tiles along rows (M)
        triton.cdiv(P, BLOCK_SIZE_COL),  # number of tiles along cols (P)
    )
    stride_A_row, stride_A_col = A.stride()
    stride_B_row, stride_B_col = B.stride()
    stride_out_row, stride_out_col = out.stride()
    # Launch kernel
    matrix_multiply[grid](
        A, B,
        stride_A_row, stride_A_col,
        stride_B_row, stride_B_col,
        out, stride_out_row, stride_out_col,
        M, N, P,  # inner dim is N
        BLOCK_SIZE_ROW=BLOCK_SIZE_ROW,
        BLOCK_SIZE_COL=BLOCK_SIZE_COL,
        BLOCK_SIZE_K=BLOCK_SIZE_K
    )

    # Make sure the kernel finished
    torch.cuda.synchronize()

    # Compare with PyTorch matmul
    ref = A @ B
    max_diff = (out - ref).abs().max().item()
    print("A =\n", A)
    print("B =\n", B)
    print("Triton out =\n", out)
    print("Torch ref =\n", ref)
    print("Max difference:", max_diff)



In [28]:
test_matmul()

CompilationError: at 20:17:
    for K in range(0, N, BLOCK_SIZE_K):
        # Set of rows to load from A (BLOCK_SIZE_ROW, N)
        offset_A_row = idx_row * BLOCK_SIZE_ROW + tl.arange(0, BLOCK_SIZE_ROW)
        offset_A_col = K + tl.arange(0, BLOCK_SIZE_K)
        A_ptrs = A_ptr + offset_A_row[:, None] * stride_A_row + offset_A_col[None,:] * stride_A_col
        # Set of rows to load from B (N, BLOCK_SIZE_COL) 
        offset_B_row = K + tl.arange(0, BLOCK_SIZE_K) 
        offset_B_col = idx_col * BLOCK_SIZE_COL + tl.arange(0, BLOCK_SIZE_COL)
        B_ptrs = B_ptr + offset_B_row[:, None] * stride_B_row + offset_B_col[None,:] * stride_B_col
        A = tl.load(A_ptrs, mask = (offset_A_row[:, None] < M) & (offset_A_col[None,:] < N))
        B = tl.load(B_ptrs, mask = (offset_B_col[None,:] < N) & (offset_B_col[None,:] < P))
        block += tl.dot(A, B)
                 ^
Input shapes should have M >= 1, N >= 1 and K >= 16